## Settings

In [1]:
## Settings 
import os
from tqdm import tqdm 
from datasets import load_dataset

# Get current working directory
base_path = os.getcwd()

# Define relative paths
data_path = os.path.join(base_path, 'data', 'WAVA') 
notations_file = os.path.join(base_path, 'data', 'SCRIPT', 'Actual_Notations', 'total_notations_actual.txt')
notations_file_2 = os.path.join(base_path, 'data', 'SCRIPT', 'Suppose_Notations', 'total_notations_suppose.txt')

output_file = 'Pinyin_notations_actual.txt'    # Output file for actual pinyin
output_file_2 = 'Pinyin_notations_suppose.txt'  # Output file for supposed pinyin

# Load dataset using HuggingFace datasets
data = load_dataset("audiofolder", data_dir=data_path)

data

Generating train split: 0 examples [00:00, ? examples/s]

ValueError: Instruction "train" corresponds to no data!

## Withdraw Pinyin Notations from LATIC L2 datasets

In [ ]:

audio_filenames = []
# Loop through the dataset with a progress bar
for item in tqdm(data['train'], desc="Extracting file names"):
    audio_path = item['audio']['path']
    audio_filename = os.path.basename(audio_path)  # Extract the file name
    audio_name = os.path.splitext(audio_filename)[0]  # Remove the .wav extension
    audio_filenames.append(audio_name)

def extract_notations(audio_filenames, notations_file, filename_output):
    # Create a dictionary to hold audio filename to text mappings
    content = {}

    # Read the transcript from the specified file
    with open(notations_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Process each line in the transcript
    for line in lines:
        # Split the line into filename and text
        parts = line.strip().split('\t', 1)  # Split into two parts: filename and text

        if len(parts) == 2:
            filename, text = parts
            # If the filename (without .wav) is in the audio filenames list, map it to the text
            if filename in audio_filenames:
                content[filename] = text

    # Write the extracted texts to the text output file
    with open(filename_output, 'w', encoding='utf-8') as text_file:
        for text in content.values():
            text_file.write(f"{text}\n")

## Result below
# It will generate two files:
# 1. Pinyin_notations_actual.txt
# 2. Pinyin_notations_suppose.txt
result = extract_notations(audio_filenames, notations_file, output_file)
result = extract_notations(audio_filenames, notations_file_2, output_file_2)

## Generate Tone notations file for actual and supposed

In [ ]:
file_suppose = 'Pinyin_notations_suppose.txt'
file_actual = 'Pinyin_notations_actual.txt'

import re
with open(file_suppose, 'r') as file:
    lines_suppose = [line.strip() for line in file.readlines()]

with open(file_actual, 'r') as file:
    lines_actual = [line.strip() for line in file.readlines()]
    

def keepToneOnly(a):
    # Use a pattern to match digits only, which represent tones in Pinyin
    pattern = re.compile(r"[^\d]")
    # Replace all non-digit characters with an empty string
    a = pattern.sub('', a)
    
    a = ' '.join(a)
    return a

lines_suppose = [keepToneOnly(line) for line in lines_suppose]
lines_actual = [keepToneOnly(line) for line in lines_actual]

## Generate Tone Notations file         ->   1 2 3 4 5 4 3 2 1 Example
# generate the output files 'tone_suppose.txt' and 'tone_actual.txt'
file_suppose = 'tone_suppose.txt'
with open(file_suppose, 'w', encoding='utf-8') as output_file:
    for line in lines_suppose:
        output_file.write(f"{line}\n")
        
file_actual = 'tone_actual.txt'
with open(file_actual, 'w', encoding='utf-8') as output_file:
    for line in lines_actual:
        output_file.write(f"{line}\n")

## Sample Testing (optional)

In [ ]:
import re 

notation1 = 'Pinyin_notations_suppose.txt'
notation2 ='Pinyin_notations_actual.txt'

with open(notation1, 'r', encoding='utf-8') as file:
    lines = [line.strip() for line in file.readlines()]
    
with open(notation2, 'r', encoding='utf-8') as file:
    lines_2 = [line.strip() for line in file.readlines()]

def remove_punctuation(input_string):
    # Updated regex pattern to include a wide range of punctuation marks and spaces
    pattern = r"[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~·•《》「」『』【】…（）、；：！？——‘’“”，‧。“”·：、/ㄟＰ|・／－〉〈─□Λ]+"
    
    # Remove the unwanted characters
    cleaned_string = re.sub(pattern, '', input_string)
    
    return cleaned_string

lines = [remove_punctuation(line) for line in lines]
lines_2 = [remove_punctuation(line) for line in lines_2]

i = 0
for i in range(len(lines)):
    if (lines[i] != lines_2[i]):
        print("Should be said            ", lines[i], "index :" , i)
        print("L2 speakers actually said ", lines_2[i])
        print("\n")
        i = i + 1
print("total number of incorrect pronuncition: ", i)

### Draw the direcotry (optional)

In [ ]:
from anytree import Node, RenderTree
import os

def build_tree(path, parent=None):
    name = os.path.basename(path)
    node = Node(name, parent=parent)
    if os.path.isdir(path):
        for item in sorted(os.listdir(path)):
            build_tree(os.path.join(path, item), node)
    return node

# Path to your LATIC root directory
root_path = base_path
tree_root = build_tree(root_path)

# Render the tree
for pre, fill, node in RenderTree(tree_root):
    print(f"{pre}{node.name}")